[Reference](https://medium.com/@Rohan_Dutt/ae9affc3c95d)

# 1. Monte Carlo Simulation (The “What-If” Machine)

In [10]:
import numpy as np
import pandas as pd

def run_monte_carlo(start_mrr, growth_mean, growth_std, churn_mean, churn_std, sims=1000, months=12):
    """
    Runs 1,000 simulations of the next 12 months using randomized market conditions.
    """
    results = []

    for i in range(sims):
        mrr = start_mrr
        path = [mrr]

        for _ in range(months):
            # Randomly sample growth and churn from a Normal Distribution
            # This simulates "good months" and "bad months"
            monthly_growth = np.random.normal(growth_mean, growth_std)
            monthly_churn = np.random.normal(churn_mean, churn_std)

            # Clamp churn to 0 (can't have negative churn in this simple model)
            monthly_churn = max(0, monthly_churn)

            # Apply changes
            mrr = mrr * (1 + monthly_growth - monthly_churn)
            path.append(mrr)

        results.append(path[-1]) # Store only the final month's MRR

    # Analyze the 1,000 outcomes
    results = np.array(results)

    return {
        "Worst_Case (10th Percentile)": round(np.percentile(results, 10), 2),
        "Base_Case (Median)":           round(np.median(results), 2),
        "Best_Case (90th Percentile)":  round(np.percentile(results, 90), 2),
        "Probability of Growth > 20%":  f"{round((np.sum(results > start_mrr * 1.2) / sims) * 100, 1)}%"
    }

# Configuration
CURRENT_MRR = 100000
AVG_GROWTH = 0.05    # 5% Average Growth
VOLATILITY_G = 0.02  # +/- 2% Volatility (Standard Deviation)
AVG_CHURN = 0.03     # 3% Average Churn
VOLATILITY_C = 0.01  # +/- 1% Volatility

risk_report = run_monte_carlo(CURRENT_MRR, AVG_GROWTH, VOLATILITY_G, AVG_CHURN, VOLATILITY_C)

print(f"--- Monte Carlo Risk Assessment (N=1,000 Sims) ---")
for k, v in risk_report.items():
    print(f"{k}: {v}")

--- Monte Carlo Risk Assessment (N=1,000 Sims) ---
Worst_Case (10th Percentile): 114537.98
Base_Case (Median): 126273.15
Best_Case (90th Percentile): 139026.2
Probability of Growth > 20%: 74.9%


# 2. Unit Economics Forecast (LTV:CAC)

In [9]:
def analyze_unit_economics(arpu, gross_margin, churn_rate, total_marketing_spend, new_customers):
    """
    Calculates LTV, CAC, and Payback Period to determine business health.

    Parameters:
    - arpu: Average Revenue Per User (monthly)
    - gross_margin: % of revenue kept after COGS (hosting, support)
    - churn_rate: % of customers lost monthly
    """

    # 1. Calculate Lifetime Value (LTV)
    # LTV = (ARPU * Margin) / Churn Rate
    gross_profit_per_user = arpu * gross_margin
    ltv = gross_profit_per_user / churn_rate if churn_rate > 0 else 0

    # 2. Calculate Customer Acquisition Cost (CAC)
    cac = total_marketing_spend / new_customers if new_customers > 0 else 0

    # 3. Calculate Ratio
    ratio = ltv / cac if cac > 0 else 0

    # 4. Calculate Payback Period (Months to recover CAC)
    payback_months = cac / gross_profit_per_user if gross_profit_per_user > 0 else 0

    # 5. Diagnostic
    health_status = "HEALTHY (Scale Now)"
    if ratio < 1:
        health_status = "CRITICAL (Losing Money per User)"
    elif ratio < 3:
        health_status = "WARNING (Optimize Churn or CAC)"

    return {
        "ARPU": arpu,
        "LTV": round(ltv, 2),
        "CAC": round(cac, 2),
        "LTV:CAC Ratio": round(ratio, 2),
        "Payback_Months": round(payback_months, 1),
        "Status": health_status
    }

# Scenario: High Spend, High Churn (The "Burnout" Model)
metrics = analyze_unit_economics(
    arpu=50,
    gross_margin=0.80,      # 80% SaaS Margin
    churn_rate=0.08,        # 8% Churn (High)
    total_marketing_spend=20000,
    new_customers=100
)

print(f"--- Unit Economics Analysis ---")
for k, v in metrics.items():
    print(f"{k}: {v}")

--- Unit Economics Analysis ---
ARPU: 50
LTV: 500.0
CAC: 200.0
LTV:CAC Ratio: 2.5
Payback_Months: 5.0
Status: WARNING (Optimize Churn or CAC)


# 3. Churn-Adjusted Model (The Silent Killer)

In [8]:
import pandas as pd
import numpy as np

def simulate_churn_impact(start_mrr, new_sales_per_month, churn_rate, months=12):
    """
    Compares revenue trajectory with and without churn to visualize lost value.
    """
    data = []

    # Initialize variables
    mrr_perfect = start_mrr # Scenario A: 0% Churn
    mrr_real = start_mrr    # Scenario B: Actual Churn

    for month in range(1, months + 1):
        # Scenario A: Pure Growth
        mrr_perfect += new_sales_per_month

        # Scenario B: Growth minus Churn
        churn_loss = mrr_real * churn_rate
        net_change = new_sales_per_month - churn_loss
        mrr_real += net_change

        # Calculate the cumulative "Gap"
        lost_revenue = mrr_perfect - mrr_real

        data.append({
            "Month": month,
            "Gross_MRR (No Churn)": round(mrr_perfect, 2),
            "Net_MRR (With Churn)": round(mrr_real, 2),
            "Monthly_Churn_Loss": round(churn_loss, 2),
            "Total_Gap": round(lost_revenue, 2)
        })

    return pd.DataFrame(data)

# Configuration
STARTING_MRR = 100000   # $100k Base
NEW_SALES = 15000       # Adding $15k new revenue/month
CHURN_RATE = 0.05       # 5% Monthly Churn

impact_report = simulate_churn_impact(STARTING_MRR, NEW_SALES, CHURN_RATE)

print(f"--- The Cost of 5% Churn over 12 Months ---")
print(impact_report[['Month', 'Net_MRR (With Churn)', 'Monthly_Churn_Loss', 'Total_Gap']].to_string(index=False))

--- The Cost of 5% Churn over 12 Months ---
 Month  Net_MRR (With Churn)  Monthly_Churn_Loss  Total_Gap
     1             110000.00             5000.00    5000.00
     2             119500.00             5500.00   10500.00
     3             128525.00             5975.00   16475.00
     4             137098.75             6426.25   22901.25
     5             145243.81             6854.94   29756.19
     6             152981.62             7262.19   37018.38
     7             160332.54             7649.08   44667.46
     8             167315.91             8016.63   52684.09
     9             173950.12             8365.80   61049.88
    10             180252.61             8697.51   69747.39
    11             186239.98             9012.63   78760.02
    12             191927.98             9312.00   88072.02


# 4. Lead Velocity Forecast (The Speedometer)

In [7]:
import pandas as pd
import numpy as np

def forecast_lead_velocity(current_leads, lvr, conversion_rate, deal_size, lag_months, periods=12):
    """
    Projects revenue based on Lead Velocity Rate (LVR) and Sales Cycle Lag.

    Parameters:
    - lvr: Monthly growth in qualified leads (decimal, e.g., 0.10 for 10%)
    - lag_months: Time from Lead Creation to Closed Won
    """
    months = np.arange(1, periods + 1)

    # 1. Project Lead Volume (Compound Growth)
    # Leads_t = Leads_0 * (1 + LVR)^t
    projected_leads = current_leads * (1 + lvr)**months

    # 2. Calculate "Potential" Revenue (Leads * Conversion * Price)
    potential_revenue = projected_leads * conversion_rate * deal_size

    # 3. Apply Sales Cycle Lag (Shift Revenue Forward)
    # Create a Series and shift it; fill initial months with 0
    realized_revenue = pd.Series(potential_revenue).shift(lag_months).fillna(0)

    df = pd.DataFrame({
        'Month': months,
        'Qualified_Leads': projected_leads.astype(int),
        'Potential_Pipeline_Value': potential_revenue.round(2),
        'Realized_Revenue (Cash)': realized_revenue.round(2)
    })

    return df

# Configuration
START_LEADS = 500       # 500 MQLs/month currently
LVR = 0.08              # Leads growing 8% MoM
CONVERSION = 0.05       # 5% convert to customers
ACV = 12000             # $12k Average Contract Value
SALES_CYCLE = 3         # 3 Month Lag

forecast = forecast_lead_velocity(START_LEADS, LVR, CONVERSION, ACV, SALES_CYCLE)

# Display: Notice Revenue is 0 for the first 3 months (Lag)
print(f"--- Lead Velocity Forecast (Lag: {SALES_CYCLE} Months) ---")
print(forecast.head(6).to_string(index=False))

--- Lead Velocity Forecast (Lag: 3 Months) ---
 Month  Qualified_Leads  Potential_Pipeline_Value  Realized_Revenue (Cash)
     1              540                 324000.00                      0.0
     2              583                 349920.00                      0.0
     3              629                 377913.60                      0.0
     4              680                 408146.69                 324000.0
     5              734                 440798.42                 349920.0
     6              793                 476062.30                 377913.6


# 5. Bottom-Up Forecast (Grassroots Accuracy)

In [6]:
import pandas as pd
import numpy as np

def forecast_segment(name, start_users, arpu, growth_rate, churn_rate, months=12):
    """
    Generates a forecast for a single specific business segment.
    """
    # Create month index
    t = np.arange(1, months + 1)

    # Calculate Net Growth Rate (Growth - Churn)
    net_rate = growth_rate - churn_rate

    # Vectorized calculation for user count
    # Users_t = Users_0 * (1 + net_rate)^t
    projected_users = start_users * (1 + net_rate)**t

    # Calculate Revenue
    projected_revenue = projected_users * arpu

    return pd.DataFrame({
        'Month': t,
        'Segment': name,
        'Users': projected_users.astype(int),
        'Revenue': projected_revenue.round(2)
    })

# Configuration: Distinct Segment Physics
segments = [
    # SMB: High volume, low price, high churn, fast growth
    {"name": "SMB", "start": 1000, "arpu": 50, "growth": 0.10, "churn": 0.05},

    # Enterprise: Low volume, high price, low churn, slow growth
    {"name": "Enterprise", "start": 50, "arpu": 2500, "growth": 0.02, "churn": 0.005}
]

# 1. Generate Individual Forecasts
all_forecasts = []
for seg in segments:
    df = forecast_segment(seg['name'], seg['start'], seg['arpu'], seg['growth'], seg['churn'])
    all_forecasts.append(df)

# 2. Combine and Aggregate
full_df = pd.concat(all_forecasts)
total_forecast = full_df.groupby('Month')[['Revenue', 'Users']].sum().reset_index()

print("--- Segment Breakdown (Month 12) ---")
print(full_df[full_df['Month'] == 12].to_string(index=False))

print("\n--- Total Consolidated Revenue (First 5 Months) ---")
print(total_forecast.head().to_string(index=False))

--- Segment Breakdown (Month 12) ---
 Month    Segment  Users   Revenue
    12        SMB   1795  89792.82
    12 Enterprise     59 149452.27

--- Total Consolidated Revenue (First 5 Months) ---
 Month   Revenue  Users
     1 179375.00   1100
     2 183903.12   1153
     3 188591.05   1209
     4 193445.75   1268
     5 198474.58   1329


# 6. ARPU Expansion Forecasting (The Upsell Engine)

In [5]:
import pandas as pd
import numpy as np

def forecast_arpu_expansion(users_start, upgrade_rates, tier_prices, months=12):
    """
    Simulates revenue growth driven by users moving to higher tiers.
    """
    # Initialize User Distribution [Basic, Pro, Enterprise]
    current_users = np.array(users_start)

    # Transition Matrix (Probability of moving From Row -> To Column)
    # Rows: Current Tier | Cols: Next Month's Tier
    # Note: Diagonal = Retention in current tier
    transition_matrix = np.array([
        [0.93, 0.05, 0.00], # Basic: 93% stay, 5% upgrade to Pro, 2% Churn (implied)
        [0.00, 0.94, 0.04], # Pro: 94% stay, 4% upgrade to Ent, 2% Churn
        [0.00, 0.00, 0.98]  # Ent: 98% stay, 2% Churn
    ])

    forecast_data = []

    for month in range(1, months + 1):
        # Calculate Revenue
        revenue = sum(current_users * tier_prices)
        total_users = sum(current_users)
        arpu = revenue / total_users if total_users > 0 else 0

        forecast_data.append({
            "Month": month,
            "Basic_Users": int(current_users[0]),
            "Pro_Users": int(current_users[1]),
            "Ent_Users": int(current_users[2]),
            "Total_Revenue": round(revenue, 2),
            "ARPU": round(arpu, 2)
        })

        # Apply Transition for Next Month (Matrix Multiplication)
        # New State = Current State * Matrix
        current_users = np.dot(current_users, transition_matrix)

    return pd.DataFrame(forecast_data)

# Configuration
INITIAL_USERS = [1000, 200, 50] # 1000 Basic, 200 Pro, 50 Enterprise
PRICES = [10, 50, 200]          # Price per tier

# Run Simulation
df_expansion = forecast_arpu_expansion(INITIAL_USERS, None, PRICES)

print(df_expansion[['Month', 'Total_Revenue', 'ARPU']].to_string(index=False))

 Month  Total_Revenue  ARPU
     1       30000.00 24.00
     2       32600.00 26.61
     3       35236.00 29.35
     4       37882.40 32.20
     5       40516.98 35.14
     6       43120.55 38.16
     7       45676.67 41.25
     8       48171.35 44.39
     9       50592.80 47.57
    10       52931.20 50.79
    11       55178.50 54.03
    12       57328.22 57.28


# 7. Pipeline-Weighted Forecast (The Sales Reality Check)

In [4]:
import pandas as pd

def calculate_weighted_pipeline(deals_data, stage_probabilities):
    """
    Calculates the Expected Value (EV) of a sales pipeline based on stage probabilities.
    """
    df = pd.DataFrame(deals_data)

    # 1. Map Stage to Probability
    # Use .map() to apply the probability dictionary to the 'Stage' column
    df['Probability'] = df['Stage'].map(stage_probabilities)

    # 2. Calculate Weighted Revenue (Deal Value * Probability)
    df['Weighted_Value'] = df['Deal_Value'] * df['Probability']

    # 3. Group by Expected Close Month
    # Ensure dates are datetime objects
    df['Close_Date'] = pd.to_datetime(df['Close_Date'])
    df['Close_Month'] = df['Close_Date'].dt.to_period('M')

    forecast = df.groupby('Close_Month')[['Deal_Value', 'Weighted_Value']].sum().reset_index()

    return forecast

# Configuration: Define your sales stages and historical win rates
STAGE_PROBS = {
    "Discovery": 0.10,       # 10% chance to close
    "Demo": 0.25,            # 25% chance
    "Proposal": 0.50,        # 50% chance
    "Negotiation": 0.80,     # 80% chance
    "Contract Sent": 0.95    # 95% chance
}

# Input: Raw CRM Data Dump
raw_deals = {
    'Deal_ID': [101, 102, 103, 104, 105],
    'Stage': ['Discovery', 'Proposal', 'Negotiation', 'Demo', 'Contract Sent'],
    'Deal_Value': [50000, 25000, 100000, 15000, 75000], # Potential Revenue
    'Close_Date': ['2024-03-15', '2024-03-20', '2024-04-10', '2024-04-05', '2024-03-28']
}

report = calculate_weighted_pipeline(raw_deals, STAGE_PROBS)

print("Pipeline Forecast by Month:")
print(report.to_string(index=False))

Pipeline Forecast by Month:
Close_Month  Deal_Value  Weighted_Value
    2024-03      150000         88750.0
    2024-04      115000         83750.0


# 8. Regression-Based Forecast (Cause & Effect Model)

In [3]:
import pandas as pd
from sklearn.linear_model import LinearRegression

def predict_multivariate_revenue(data, new_ad_spend, new_sales_reps):
    """
    Predicts revenue based on multiple drivers: Ad Spend and Sales Rep Count.
    """
    df = pd.DataFrame(data)

    # Independent Variables (Features)
    X = df[['ad_spend', 'sales_reps']]
    # Dependent Variable (Target)
    y = df['revenue']

    # Train Model
    model = LinearRegression()
    model.fit(X, y)

    # Predict for new scenario
    prediction = model.predict([[new_ad_spend, new_sales_reps]])[0]

    # Extract "Impact Coefficients" (The "Power" of each variable)
    impact = {
        "Base_Revenue": round(model.intercept_, 2),
        "Impact_per_$1_Ad_Spend": round(model.coef_[0], 2),
        "Impact_per_Sales_Rep": round(model.coef_[1], 2),
        "Predicted_Revenue": round(prediction, 2)
    }

    return impact

# Historical Data (Training Set)
dataset = {
    'ad_spend':   [1000, 1500, 2000, 2500, 3000, 1200, 4000],
    'sales_reps': [2,    2,    3,    3,    4,    2,    5],
    'revenue':    [5000, 6200, 8500, 9800, 12500, 5500, 16000]
}

# Scenario: What if we spend $5k on ads and hire 6 reps?
NEW_BUDGET = 5000
NEW_HEADCOUNT = 6

results = predict_multivariate_revenue(dataset, NEW_BUDGET, NEW_HEADCOUNT)

print(f"Scenario Prediction: ${results['Predicted_Revenue']}")
print(f"Insight: Each new Sales Rep adds approx ${results['Impact_per_Sales_Rep']} to MRR.")
print(f"Insight: Every $1 in Ads adds approx ${results['Impact_per_$1_Ad_Spend']} to MRR.")

Scenario Prediction: $19712.16
Insight: Each new Sales Rep adds approx $1313.41 to MRR.
Insight: Every $1 in Ads adds approx $2.37 to MRR.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


# 9. Cohort-Based Forecasting (Where Retention Rules)

In [2]:
import pandas as pd
import numpy as np

def generate_cohort_forecast(df, forecast_months=6):
    """
    Inputs: df with columns ['user_id', 'invoice_date', 'revenue']
    """
    # 1. Create Cohort Month (First Purchase)
    df['invoice_month'] = df['invoice_date'].dt.to_period('M')
    df['cohort_month'] = df.groupby('user_id')['invoice_date'].transform('min').dt.to_period('M')

    # 2. Calculate Cohort Index (Months since acquisition)
    df['cohort_index'] = (df['invoice_month'].view(int) - df['cohort_month'].view(int))

    # 3. Create Pivot Table for Revenue Retention
    cohort_pivot = df.pivot_table(index='cohort_month',
                                  columns='cohort_index',
                                  values='revenue',
                                  aggfunc='sum').fillna(0)

    # 4. Calculate Average Decay Rate per Month
    # (Sum of revenue in Index N / Sum of revenue in Index 0)
    retention_rates = cohort_pivot.sum() / cohort_pivot.iloc[:, 0].sum()

    # 5. Forecast Future Revenue for a New Cohort
    starting_revenue = 50000  # Example: Expected revenue from next month's new users
    forecast = [starting_revenue * rate for rate in retention_rates[:forecast_months]]

    return cohort_pivot, retention_rates, forecast

# Example Usage with Dummy Data
data = {
    'user_id': np.random.randint(1, 100, 500),
    'invoice_date': pd.to_datetime(np.random.choice(pd.date_range('2024-01-01', '2024-12-31'), 500)),
    'revenue': np.random.uniform(10, 100, 500)
}
df_sample = pd.DataFrame(data)
pivot, rates, future_projection = generate_cohort_forecast(df_sample)

print("Average Revenue Retention Rates by Month Index:")
print(rates.head().round(4))

Average Revenue Retention Rates by Month Index:
cohort_index
0    1.0000
1    0.2170
2    0.3907
3    0.2932
4    0.3545
dtype: float64


/tmp/ipython-input-405348835.py:13: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  df['cohort_index'] = (df['invoice_month'].view(int) - df['cohort_month'].view(int))


# 10. Straight-Line Forecast (Simple)

In [1]:
import pandas as pd
import numpy as np

def calculate_straight_line_forecast(current_mrr, monthly_growth_rate, churn_rate, periods=12):
    """
    Calculates revenue projection using a net growth linear approach.
    """
    # Calculate net multiplier (Growth - Churn)
    net_growth = monthly_growth_rate - churn_rate

    # Generate month indices
    months = np.arange(1, periods + 1)

    # Vectorized projection: MRR_t = MRR_0 * (1 + net_growth)^t
    forecast_values = current_mrr * (1 + net_growth)**months

    forecast_df = pd.DataFrame({
        'Month': months,
        'Projected_MRR': forecast_values.round(2),
        'Growth_Delta': (forecast_values - current_mrr).round(2)
    })

    return forecast_df

# Parameters
CURRENT_MRR = 100000  # $100k
GROWTH_EXPECTED = 0.08 # 8%
CHURN_BUFFER = 0.02   # 2%
DURATION = 6          # 6 Months

results = calculate_straight_line_forecast(CURRENT_MRR, GROWTH_EXPECTED, CHURN_BUFFER, DURATION)
print(results.to_string(index=False))

 Month  Projected_MRR  Growth_Delta
     1      106000.00       6000.00
     2      112360.00      12360.00
     3      119101.60      19101.60
     4      126247.70      26247.70
     5      133822.56      33822.56
     6      141851.91      41851.91
